In [ ]:
#| default_exp foundation

## Foundation

Small shared mechanics: CLI behavior, failure telemetry, cell parsing, validation, chapter navigation, and dynamic cell classification for notebook context.

This notebook is the shared basement of the project. It keeps the small private helpers that every public tool relies on: CLI behavior, safe error reporting, cell selection, cell parsing, semantic cell classification, and chapter addressing.

Most functions here are intentionally private. The rest of the project can stay user-facing because this notebook absorbs the messy details of notebooks as data structures.

Read it as a tour of the shapes nbskill needs to recognize. A notebook is not just text: it has cells, headings, generated Python files, visible outputs, and tiny bits of metadata. The foundation layer gives all later notebooks one vocabulary for those shapes, so the public tools can talk about notebooks without making users think in raw JSON.

### Production contract

This notebook owns the shared invariants for every other tool. Production behavior is: path helpers resolve notebooks from project-root and notebook-dir execution, cell-block parsing creates valid notebook cells, metadata stamping records semantic classes and export hashes, and validation reports missing or stale metadata before edits proceed.


The helpers here deliberately stay small because every higher-level tool depends on them. For example, `parse_cells` turns friendly cell-block text into notebook cells, the semantic classifiers let readers say "show me tests" or "show me exported code" without parsing raw notebook JSON.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import os

from fastcore.test import test_eq

In [ ]:
#| export
import ast,hashlib,json,multiprocessing,os,queue,re,tempfile,tomllib,traceback
import shutil,subprocess,sys
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from dataclasses import dataclass, field
from enum import Enum
from importlib.resources import files
from io import StringIO
from pathlib import Path
from fastcore.basics import in_jupyter, patch
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.script import is_cli
from fastcore.xtras import rtoken_hex

### Demo scratch files

Examples and tests in later notebooks need small notebooks they can safely mutate. Scratch helpers create disposable artifacts under `nbs/data` and remove them after use. The checked-in tool fixture lives at `tests/fixtures/nbskill_tool_fixture.ipynb`, and `revert_example_notebook` rewrites it to a known baseline whenever a test or demo mutates it.

In [ ]:
#| export
def remove_demo_path(path):
    path = Path(path)
    if path.is_dir(): shutil.rmtree(path)
    elif path.exists(): path.unlink()
    return path

`demo_path` names a scratch artifact and optionally removes any previous copy. It only creates the parent folder; helpers that write content decide what belongs at the path.

Use `demo_path_context` when a test needs an arbitrary scratch file or directory and should clean it up after the block. That keeps cleanup policy in one helper instead of repeating `try`/`finally` in every test.

In [ ]:
#| export
def demo_path(name, base="nbs/data", reset=True):
    'Return a scratch path under `base`, optionally removing any previous artifact.'
    path = Path(base) / name
    if reset: remove_demo_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


@contextmanager
def demo_path_context(name, base="nbs/data", reset=True):
    'Yield a demo path and remove it when the block exits.'
    path = demo_path(name, base=base, reset=reset)
    try: yield path
    finally: remove_demo_path(path)

In [ ]:
#| hide
with demo_path_context("foundation_demo_path_context") as context_path:
    context_seen = context_path
assert context_seen == Path("nbs/data/foundation_demo_path_context")

In [ ]:
#| export
def path_candidates(path):
    "Return cwd-relative path candidates for root and notebook-dir execution."
    pth = Path(str(path)).expanduser()
    if pth.is_absolute(): return [pth]
    parts = pth.parts
    in_notebook_dir = parts and parts[0] == Path.cwd().name and (in_jupyter() or Path.cwd().name == "nbs")
    if not in_notebook_dir: return [pth]
    stripped = Path(*parts[1:]) if len(parts) > 1 else Path(".")
    return [pth, stripped]

In [ ]:
_path_candidate_rows = path_candidates("nbs/00_foundation.ipynb")
assert _path_candidate_rows[0] == Path("nbs/00_foundation.ipynb")
if Path.cwd().name == "nbs":
    assert _path_candidate_rows[-1] == Path("00_foundation.ipynb")
else:
    assert _path_candidate_rows == [Path("nbs/00_foundation.ipynb")]
assert path_candidates(Path("/tmp/example.ipynb")) == [Path("/tmp/example.ipynb")]
print("path_candidates handles root and notebook-dir paths")

path_candidates handles root and notebook-dir paths


Notebook helpers should fail gently when a path is missing, unreadable, or shaped like ordinary JSON instead of a Jupyter notebook. `is_valid_ipynb` keeps that contract small and dependency-free.

In [ ]:
#| export
_VALID_CELL_TYPES = {"code", "markdown", "raw"}


def _is_valid_notebook_cell(cell):
    if not isinstance(cell, dict): return False
    if cell.get("cell_type") not in _VALID_CELL_TYPES: return False
    if not isinstance(cell.get("metadata"), dict): return False
    source = cell.get("source")
    if not isinstance(source, (str, list)): return False
    if isinstance(source, list) and not all(isinstance(line, str) for line in source): return False
    if cell["cell_type"] == "code":
        if "outputs" not in cell or not isinstance(cell.get("outputs"), list): return False
        execution_count = cell.get("execution_count")
        if execution_count is not None and not isinstance(execution_count, int): return False
    return True

In [ ]:
#| export
def is_valid_ipynb(path):
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        return False
    if not isinstance(data, dict): return False
    if not isinstance(data.get("nbformat"), int): return False
    if not isinstance(data.get("nbformat_minor"), int): return False
    if not isinstance(data.get("metadata"), dict): return False
    cells = data.get("cells")
    return isinstance(cells, list) and all(_is_valid_notebook_cell(cell) for cell in cells)

`write_demo_notebook` owns the whole lifetime of a scratch notebook: create it, yield its path to the example, then remove it even if the example fails.

In [ ]:
#| export
@contextmanager
def write_demo_notebook(name, cells=None, base="nbs/data", reset=True):
    path = demo_path(name, base=base, reset=reset)
    cells = cells or [
        mk_cell("## Demo notebook\nThis tiny notebook gives nbskill tools something real to inspect.", cell_type="markdown"),
        mk_cell("#| export\ndef demo_answer():\n    return 42"),
        mk_cell("assert demo_answer() == 42")]
    nb = new_nb(cells)
    write_nb(nb, path)
    try:
        yield path
    finally:
        remove_demo_path(path)

The shared tool fixture is deliberately checked in. It gives readers, editors, executors, reviewers, and MCP wrappers one stable notebook shape to exercise, while `revert_example_notebook` makes the file reusable after mutation-heavy examples.

In [ ]:
#| export
EXAMPLE_NOTEBOOK_PATH = Path("tests/fixtures/nbskill_tool_fixture.ipynb")


def example_notebook():
    "Return the canonical notebook used by tool tests and examples."
    cells = [
        mk_cell(
            "# nbskill tool fixture\n\nA tiny checked-in notebook for reversible tool tests.",
            cell_type="markdown",
            id="fixture-title",
        ),
        mk_cell("answer = 42", id="fixture-answer"),
        mk_cell("from fastcore.test import test_eq", id="fixture-imports"),
        mk_cell(
            "## Arithmetic\n\n`double_answer` gives readers and symbol tools a public function to find.",
            cell_type="markdown",
            id="fixture-arithmetic-doc",
        ),
        mk_cell(
            "#| export\ndef double_answer(value):\n    \"Return twice the input value.\"\n    return value * 2",
            id="fixture-double-answer",
        ),
        mk_cell("double_answer(answer)", id="fixture-example"),
        mk_cell("test_eq(double_answer(21), 42)", id="fixture-test"),
    ]
    nb = new_nb(cells)
    nb.metadata["nbskill"] = {"fixture": "tool", "version": 1}
    return nb


def revert_example_notebook(path=EXAMPLE_NOTEBOOK_PATH):
    "Rewrite the shared example notebook to its canonical fixture state."
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_nb(example_notebook(), path)
    return path

In [ ]:
#| hide
from fastcore.nbio import read_nb

example_ids = [cell.id for cell in example_notebook().cells]
test_eq(example_ids[:3], ["fixture-title", "fixture-answer", "fixture-imports"])
test_eq(example_ids[-1], "fixture-test")

fixture = demo_path("fixture_revert.ipynb")
try:
    path = revert_example_notebook(fixture)
    nb = read_nb(path)
    test_eq(path, fixture)
    test_eq([cell.id for cell in nb.cells][:3], ["fixture-title", "fixture-answer", "fixture-imports"])

    nb.cells[1].source = "answer = 7"
    write_nb(nb, path)
    revert_example_notebook(path)
    test_eq(read_nb(path).cells[1].source, "answer = 42")
finally:
    remove_demo_path(fixture)

Here is the tiny notebook fixture in the wild. It gives examples a real `.ipynb` to read, then vanishes after the context exits, which keeps the repo tidy while still letting the rendered notebook show concrete output.

In [ ]:
from fastcore.nbio import read_nb

with write_demo_notebook("00_foundation_tour.ipynb") as path:
    nb = read_nb(path)
    print(path.name)
    print([cell.cell_type for cell in nb.cells])
    print(str(nb.cells[0].source).splitlines()[0])

print(path.exists())

00_foundation_tour.ipynb
['markdown', 'code', 'code']
## Demo notebook
False


In [ ]:
with write_demo_notebook("00_foundation_bad.ipynb") as bad_path:
    bad_path.write_text("{}", encoding="utf-8")
    assert not is_valid_ipynb(bad_path)
    assert not is_valid_ipynb(bad_path.with_suffix(".missing.ipynb"))

### CLI contracts and diagnostics

The public functions in later notebooks are both Python functions and command-line commands. These helpers make that dual use predictable: direct Python calls return values, CLI calls print user-friendly errors, and tool starts or failures are recorded in a small local failure map for debugging repeated friction.

In [ ]:
#| export
def cli_return(value=None):
    return None if is_cli() else value

In [ ]:
#| export
def cli_error(msg):
    if is_cli():
        print(msg, file=sys.stderr)
        raise SystemExit(1)
    raise ValueError(msg)

In [ ]:
#| export
def failure_map_path():
    default = Path.home() / ".nbskill-errors.json"
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", default)).expanduser()


In [ ]:
#| export
def empty_failure_map():
    return {"version": 1, "events": [], "counts": {}, "last_call": None}


In [ ]:
#| export
def load_failure_map(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        data = empty_failure_map()
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

In [ ]:
#| export
_NBSKILL_HOOKS_MARKER_START = "# nbskill nbdev hooks:start"


In [ ]:
#| export
_NBSKILL_HOOKS_MARKER_END = "# nbskill nbdev hooks:end"


In [ ]:
#| export
_NBSKILL_HOOKS_INSTALLED_SENTINEL = "nbskill-hooks-installed"


In [ ]:
#| export
_NBSKILL_HOOKS_DISABLED_SENTINEL = "nbskill-hooks-disabled"


In [ ]:
#| export
def git_run(root, *args):
    "Run a git command from `root` and return stdout, or empty text on failure."
    proc = subprocess.run(["git", "-C", str(root), *args], text=True, capture_output=True)
    return proc.stdout if proc.returncode == 0 else ""


def git_root(path="."):
    "Return the containing git repository root, if one exists."
    path = Path(path).expanduser()
    base = path.parent if path.suffix else path
    candidates = [base, *path_candidates(base), Path(".")]
    for candidate in dict.fromkeys(candidates):
        out = git_run(candidate, "rev-parse", "--show-toplevel").strip()
        if out: return Path(out)
    return None

In [ ]:
#| export
def git_status_paths(root):
    "Return paths with working-tree changes relative to `root`."
    paths = set()
    for line in git_run(root, "status", "--porcelain").splitlines():
        raw = line[3:].strip() if len(line) > 3 else line.strip()
        if " -> " in raw: raw = raw.split(" -> ", 1)[1]
        if raw: paths.add(raw)
    return paths


def git_tracked_paths(root):
    "Return git-tracked paths relative to `root`."
    return set(git_run(root, "ls-files").splitlines())

In [ ]:
#| export
def git_diff_stats(root, ref="HEAD"):
    "Return added/deleted line stats for working-tree changes from `ref`."
    stats = {}
    for line in git_run(root, "diff", "--numstat", ref, "--").splitlines():
        added, deleted, path = line.split("\t", 2)
        stats[path] = {
            "added": int(added) if added.isdigit() else 0,
            "deleted": int(deleted) if deleted.isdigit() else 0,
        }
    return stats

In [ ]:
#| export
def _is_notebook_path(path):
    path = Path(path)
    hidden = any(part.startswith(".") or part == ".ipynb_checkpoints" for part in path.parts)
    return path.suffix == ".ipynb" and not hidden and not path.name.startswith("_")


def _has_glob_chars(path):
    return any(char in str(path) for char in "*?[]")

In [ ]:
#| export
from fastcore.basics import AttrDict

_DEFAULT_CONFIG = AttrDict(exclude_dirs=["data", "dev", ".ipynb_checkpoints"])

def load_nbskill_config(path="."):
    "Load .nbskill.py config from project root, merged over defaults."
    cfg = AttrDict(**_DEFAULT_CONFIG)
    root = git_root(path)
    if root is None: root = Path(path).resolve()
    cfg_path = Path(root) / ".nbskill.py"
    if cfg_path.exists():
        ns = {}
        exec(cfg_path.read_text(encoding="utf-8"), ns)
        cfg.update({k: v for k, v in ns.items() if not k.startswith("_")})
    return cfg

In [ ]:
#| export
def _glob_notebook_paths(path):
    try:
        from fastcore.xtras import globtastic
        cfg = load_nbskill_config(path)
        skip_re = r"(^[_.]|" + "|".join(re.escape(d) for d in cfg.exclude_dirs) + r")"
        if _has_glob_chars(path):
            import glob
            return [Path(item) for item in glob.glob(str(path), recursive=True)]
        return [Path(item) for item in globtastic(path, file_glob="*.ipynb", skip_folder_re=skip_re, skip_file_re=r"^[_.]", func=Path)]
    except Exception:
        return sorted(Path(path).rglob("*.ipynb")) if Path(path).is_dir() else []


def _nbdev_notebook_paths(path):
    try:
        from nbdev.doclinks import nbglob
        return [Path(item) for item in nbglob(path=path, as_path=True)]
    except Exception:
        return []

In [ ]:
#| export
def notebook_paths(path="nbs"):
    "Return visible notebook paths for a file, directory, glob, or nbdev project path."
    raw = "." if path is None else str(path)
    cfg = load_nbskill_config(raw)
    exclude = set(cfg.exclude_dirs)
    base = Path(raw).expanduser().resolve()
    def _excluded(p):
        p = Path(p).expanduser()
        if not p.is_absolute():
            p = p.resolve() if p.exists() else (base / p).resolve()
        try: rel = p.relative_to(base)
        except ValueError: return False
        return any(part in exclude for part in rel.parts)
    for candidate in path_candidates(raw):
        pth = Path(candidate).expanduser()
        if pth.is_file(): paths = [pth]
        elif _has_glob_chars(raw): paths = _glob_notebook_paths(pth)
        elif pth.is_dir(): paths = _nbdev_notebook_paths(pth) or _glob_notebook_paths(pth)
        else: paths = []
        paths = sorted({item for item in paths if _is_notebook_path(item) and not _excluded(item)})
        if paths: return paths
    return []

In [ ]:
#| export
def source_without_directives(source):
    "Return source with nbdev cell directives removed."
    return "\n".join(line for line in str(source or "").splitlines() if not line.lstrip().startswith("#|"))

In [ ]:
#| export
def source_hash(source, length=12):
    """Return a SHA-256 source hash, shortened unless `length` is `None`."""
    digest = hashlib.sha256(str(source).encode('utf-8')).hexdigest()
    return digest if length is None else digest[:int(length)]

In [ ]:
#| export
def file_line_count(path):
    "Return the number of text lines in `path`, or zero when it cannot be read."
    try: return len(Path(path).read_text(encoding="utf-8", errors="ignore").splitlines())
    except OSError: return 0


def cap_text(text, limit=2000, max_output_chars=None):
    "Cap text, returning a string or review-style metadata when `max_output_chars` is used."
    metadata = max_output_chars is not None
    limit = max_output_chars if metadata else limit
    text = "" if text is None else str(text)
    if limit is None or len(text) <= limit:
        capped, omitted = text, 0
    else:
        omitted = len(text) - limit
        capped = f"{text[:limit].rstrip()}\n... truncated {omitted} chars ..."
    if not metadata: return capped
    return {"text": capped, "truncated": bool(omitted), "chars": len(text), "omitted_chars": omitted}

In [ ]:
#| export
_GENERATED_HEADER_RE = re.compile(r"^# AUTOGENERATED! DO NOT EDIT! File to edit: (.+)$")


def _generated_owner_from_header(path):
    try: lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()[:3]
    except OSError: return None
    for line in lines:
        match = _GENERATED_HEADER_RE.match(line.strip())
        if match: return (Path(path).parent / match.group(1).rstrip(".")).resolve()
    return None


def generated_owner(path, notebooks=None):
    "Return the source notebook for an nbdev-generated Python file, when known."
    path = Path(path)
    if path.suffix != ".py" or not path.exists(): return None
    owner = _generated_owner_from_header(path)
    if owner is not None or notebooks is None: return owner
    target = path.resolve()
    for nb_path in notebook_paths(notebooks):
        try: py_path = exported_py_path(nb_path)
        except (FileNotFoundError, OSError): py_path = None
        if py_path is not None and Path(py_path).exists() and Path(py_path).resolve() == target:
            return Path(nb_path).resolve()
    return None

In [ ]:
#| export
def _looks_like_nbdev_project(root):
    root = Path(root)
    pyproject = root / "pyproject.toml"
    if (root / "nbs").exists() or (root / "settings.ini").exists(): return True
    return pyproject.exists() and "[tool.nbdev]" in pyproject.read_text(encoding="utf-8", errors="ignore")


In [ ]:
#| export
def _nbdev_hook_block():
    return f"""{_NBSKILL_HOOKS_MARKER_START}
run_nbdev_cmd() {{
  if command -v "$1" >/dev/null 2>&1; then
    "$@"
  elif command -v uv >/dev/null 2>&1; then
    uv run "$@"
  else
    echo "nbskill: missing $1; install nbdev or uv" >&2
    exit 127
  fi
}}

run_nbdev_cmd nbdev-clean
if ! git diff --quiet -- .; then
  echo "nbskill: nbdev-clean changed notebooks. Review and stage those changes before committing." >&2
  exit 1
fi
run_nbdev_cmd nbdev-test --n_workers 0
{_NBSKILL_HOOKS_MARKER_END}
"""


In [ ]:
#| export
def _replace_marked_block(text, block):
    if _NBSKILL_HOOKS_MARKER_START in text and _NBSKILL_HOOKS_MARKER_END in text:
        before = text.split(_NBSKILL_HOOKS_MARKER_START, 1)[0].rstrip()
        after = text.split(_NBSKILL_HOOKS_MARKER_END, 1)[1].lstrip()
        return f"{before}\n\n{block}\n{after}".rstrip() + "\n"
    prefix = text.rstrip() if text.strip() else "#!/bin/sh"
    return f"{prefix}\n\n{block}\n"


In [ ]:
#| export
def _hook_state_paths(root):
    info = Path(root) / ".git" / "info"
    return info / _NBSKILL_HOOKS_INSTALLED_SENTINEL, info / _NBSKILL_HOOKS_DISABLED_SENTINEL


In [ ]:
#| export
def _has_nbskill_hook_block(pre_commit):
    if not pre_commit.exists(): return False
    text = pre_commit.read_text(encoding="utf-8", errors="ignore")
    return _NBSKILL_HOOKS_MARKER_START in text and _NBSKILL_HOOKS_MARKER_END in text


In [ ]:
#| export
def _hooks_removed_by_user(root, pre_commit):
    installed, disabled = _hook_state_paths(root)
    if disabled.exists(): return True
    if installed.exists() and not _has_nbskill_hook_block(pre_commit):
        disabled.write_text("nbskill hooks were removed; not reinstalling automatically\n", encoding="utf-8")
        return True
    return False


In [ ]:
#| export
def install_nbdev_pre_commit_hooks(path=".", run_nbdev_install_hooks=True):
    "Install nbdev-clean and nbdev-test pre-commit hooks in a git-backed nbdev project."
    root = git_root(path)
    if root is None: return {"installed": False, "reason": "not-a-git-repo"}
    if not _looks_like_nbdev_project(root): return {"installed": False, "reason": "not-an-nbdev-project", "root": str(root)}
    hooks = root / ".git" / "hooks"
    pre_commit = hooks / "pre-commit"
    if _hooks_removed_by_user(root, pre_commit): return {"installed": False, "reason": "hooks-removed-by-user", "root": str(root)}
    if run_nbdev_install_hooks:
        cmd = ["nbdev-install-hooks"] if shutil.which("nbdev-install-hooks") else None
        if cmd is None and shutil.which("uv"): cmd = ["uv", "run", "nbdev-install-hooks"]
        if cmd is not None: subprocess.run(cmd, cwd=root, text=True, capture_output=True)
    hooks.mkdir(parents=True, exist_ok=True)
    text = pre_commit.read_text(encoding="utf-8", errors="ignore") if pre_commit.exists() else ""
    pre_commit.write_text(_replace_marked_block(text, _nbdev_hook_block()), encoding="utf-8")
    pre_commit.chmod(pre_commit.stat().st_mode | 0o111)
    installed, _ = _hook_state_paths(root)
    installed.parent.mkdir(parents=True, exist_ok=True)
    installed.write_text("nbskill hooks installed once\n", encoding="utf-8")
    return {"installed": True, "root": str(root), "hook": str(pre_commit)}

The MCP server may be launched from many notebook projects, so it needs a small idempotent project bootstrap. Once a client workspace root is known, `bootstrap_nbskill_project` can copy the shared agent instructions, make notebook CI serial, and preserve nbskill metadata through nbdev cleanups without clobbering local `AGENTS.md` edits.

In [ ]:
#| export
_NBSKILL_PROJECT_BOOTSTRAP_ENV = "NBSKILL_NO_PROJECT_BOOTSTRAP"
_NBSKILL_AGENTS_TEMPLATE = "AGENTS.md"
_NBSKILL_CI_PAUSE = "1"
_NBSKILL_NBDEV_CONFIG = {
    "allowed_metadata_keys": '["nbskill"]',
    "jupyter_hooks": "true",
    "allowed_cell_metadata_keys": '["nbskill"]',
}


def _project_root_path(path):
    root = Path(path).expanduser()
    if root.is_file(): root = root.parent
    return root.resolve()


def _read_nbskill_agents_template():
    try:
        return files("nbskill").joinpath(_NBSKILL_AGENTS_TEMPLATE).read_text(encoding="utf-8")
    except (FileNotFoundError, ModuleNotFoundError, OSError):
        pass
    fallback = Path(__file__).with_name(_NBSKILL_AGENTS_TEMPLATE) if "__file__" in globals() else None
    for path in (fallback, Path(_NBSKILL_AGENTS_TEMPLATE)):
        if path is None: continue
        try:
            if path.exists(): return path.read_text(encoding="utf-8")
        except OSError:
            continue
    return None


def _install_agents_md(root, overwrite=False):
    path = root / "AGENTS.md"
    if path.exists() and not overwrite:
        return {"changed": False, "path": str(path), "reason": "exists"}
    text = _read_nbskill_agents_template()
    if text is None:
        return {"changed": False, "path": str(path), "reason": "missing-template"}
    try:
        current = path.read_text(encoding="utf-8") if path.exists() else None
    except OSError:
        current = None
    if current == text:
        return {"changed": False, "path": str(path), "reason": "current"}
    path.write_text(text, encoding="utf-8")
    reason = "installed" if current is None else "updated"
    return {"changed": True, "path": str(path), "reason": reason}

In [ ]:
#| export
def _set_cli_option(args, name, value):
    pattern = re.compile(rf"(^|\s){re.escape(name)}(?:=|\s+)\S+")
    if pattern.search(args):
        return pattern.sub(lambda match: f"{match.group(1)}{name} {value}", args, count=1)
    return f"{args} {name} {value}"


def _nbdev_test_serial_command(text, pause=_NBSKILL_CI_PAUSE):
    pattern = re.compile(r"(?P<cmd>\b(?:uv\s+run\s+)?nbdev-test\b)(?P<args>[^\n#]*)")

    def repl(match):
        raw_args = match.group("args")
        args = raw_args.rstrip()
        trailing = raw_args[len(args):]
        args = _set_cli_option(args, "--n_workers", "0")
        args = _set_cli_option(args, "--pause", str(pause))
        return f"{match.group('cmd')}{args}{trailing}"

    return pattern.sub(repl, text)


def _patch_github_actions(root, pause=_NBSKILL_CI_PAUSE):
    workflow_dir = root / ".github" / "workflows"
    paths = sorted([*workflow_dir.glob("*.yml"), *workflow_dir.glob("*.yaml")]) if workflow_dir.exists() else []
    results = []
    for path in paths:
        text = path.read_text(encoding="utf-8")
        updated = _nbdev_test_serial_command(text, pause=pause)
        changed = updated != text
        if changed: path.write_text(updated, encoding="utf-8")
        reason = "patched" if changed else "current"
        results.append({"changed": changed, "path": str(path), "reason": reason})
    return results

In [ ]:
#| export
def _toml_section_bounds(lines, section):
    header = f"[{section}]"
    start = next((idx for idx, line in enumerate(lines) if line.strip() == header), None)
    if start is None: return None, None
    end = len(lines)
    for idx in range(start + 1, len(lines)):
        stripped = lines[idx].strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            end = idx
            break
    return start, end


def _set_toml_section_keys(text, section, values):
    lines = text.splitlines()
    start, end = _toml_section_bounds(lines, section)
    if start is None:
        if lines and lines[-1].strip(): lines.append("")
        lines.append(f"[{section}]")
        start, end = len(lines) - 1, len(lines)
    for key, value in values.items():
        pattern = re.compile(rf"^\s*{re.escape(key)}\s*=")
        found = next((idx for idx in range(start + 1, end) if pattern.match(lines[idx])), None)
        line = f"{key} = {value}"
        if found is None:
            lines.insert(end, line)
            end += 1
        elif lines[found] != line:
            lines[found] = line
    return "\n".join(lines) + "\n"


def _patch_pyproject_nbdev(root):
    path = root / "pyproject.toml"
    if not path.exists():
        return {"changed": False, "path": str(path), "reason": "missing"}
    text = path.read_text(encoding="utf-8")
    updated = _set_toml_section_keys(text, "tool.nbdev", _NBSKILL_NBDEV_CONFIG)
    try:
        tomllib.loads(updated)
    except tomllib.TOMLDecodeError as exc:
        return {"changed": False, "path": str(path), "reason": f"invalid-toml: {exc}"}
    changed = updated != text
    if changed: path.write_text(updated, encoding="utf-8")
    reason = "patched" if changed else "current"
    return {"changed": changed, "path": str(path), "reason": reason}

In [ ]:
#| export
def bootstrap_nbskill_project(path=".", overwrite_agents=False, pause=_NBSKILL_CI_PAUSE):
    "Install nbskill project guardrails once a workspace root is known."
    if os.environ.get(_NBSKILL_PROJECT_BOOTSTRAP_ENV):
        return {"changed": False, "root": str(path), "reason": "disabled"}
    root = _project_root_path(path)
    if not root.exists():
        return {"changed": False, "root": str(root), "reason": "missing-root"}
    if not _looks_like_nbdev_project(root):
        return {"changed": False, "root": str(root), "reason": "not-an-nbdev-project"}
    agents = _install_agents_md(root, overwrite=overwrite_agents)
    pyproject = _patch_pyproject_nbdev(root)
    workflows = _patch_github_actions(root, pause=pause)
    changed = agents.get("changed", False) or pyproject.get("changed", False)
    changed = changed or any(item.get("changed", False) for item in workflows)
    return {
        "changed": changed,
        "root": str(root),
        "agents": agents,
        "pyproject": pyproject,
        "workflows": workflows,
    }

In [ ]:
#| hide
bootstrap_root = demo_path("00_project_bootstrap")
try:
    bootstrap_root.mkdir()
    (bootstrap_root / "nbs").mkdir()
    (bootstrap_root / "pyproject.toml").write_text(
        '[project]\nname = "demo"\n\n[tool.nbdev]\nallowed_metadata_keys = []\n',
        encoding="utf-8",
    )
    workflow = bootstrap_root / ".github" / "workflows" / "test.yaml"
    workflow.parent.mkdir(parents=True)
    workflow.write_text(
        "jobs:\n  test:\n    steps:\n      - run: uv run nbdev-test --n_workers 4 --pause 0.01\n",
        encoding="utf-8",
    )
    result = bootstrap_nbskill_project(bootstrap_root)
    assert result["changed"] is True
    assert (bootstrap_root / "AGENTS.md").exists()
    pyproject_text = (bootstrap_root / "pyproject.toml").read_text(encoding="utf-8")
    assert 'allowed_metadata_keys = ["nbskill"]' in pyproject_text
    assert "jupyter_hooks = true" in pyproject_text
    assert 'allowed_cell_metadata_keys = ["nbskill"]' in pyproject_text
    workflow_text = workflow.read_text(encoding="utf-8")
    assert "uv run nbdev-test --n_workers 0 --pause 1" in workflow_text
    second = bootstrap_nbskill_project(bootstrap_root)
    assert second["changed"] is False
finally:
    remove_demo_path(bootstrap_root)

### Parsing user-facing selectors

Selectors are the boundary between human shorthand and list operations. A caller might ask for `3`, `2:5`, `-1`, or `None`; these helpers normalize that text before any read or write tool touches notebook cells. Keeping the grammar here makes higher-level tools predictable and keeps cell-index mistakes from spreading.

In [ ]:
#| export
def parse_literal(value):
    if value is None: return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in {"", "none", "null"}: return None
        try: return ast.literal_eval(value)
        except (SyntaxError, ValueError): return value
    return value

In [ ]:
#| export
def none_if_string(value):
    return None if isinstance(value, str) and value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _parse_slice(value):
    if not isinstance(value, str) or ":" not in value: return None
    parts = value.split(":")
    if len(parts) not in (2, 3): return None
    vals = [int(p) if p else None for p in parts]
    return slice(*vals)

In [ ]:
#| export
def _as_index(value, length):
    idx = int(value)
    if idx < 0: idx += length
    if idx < 0 or idx >= length: raise IndexError(value)
    return idx

In [ ]:
#| export
def _parse_read_selector(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)): return [int(o) for o in value]
    return int(value)

In [ ]:
#| export
def _parse_write_target(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)):
        if len(value) != 2: raise ValueError("write ranges must have start and stop")
        return slice(value[0], value[1])
    return int(value)

### Turning text into notebook cells

Write tools accept plain text because that is what agents and humans can produce quickly. This section turns that text into real cells: split on block separators, honor `%%markdown`/`%%code` markers, and split exported code into symbol-sized chunks when needed. It is the small parser that makes notebook edits feel like editing a document instead of assembling JSON.

In [ ]:
#| export
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [ ]:
#| export
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

In [ ]:
#| export
def symbol_short_name(symbol):
    "Return the final dotted component of a symbol name."
    return str(symbol).rsplit(".", 1)[-1]

In [ ]:
#| export
def _name_parts(node):
    if isinstance(node, ast.Name): return [node.id]
    if isinstance(node, ast.Attribute):
        base = _name_parts(node.value)
        return [*base, node.attr] if base else [node.attr]
    return []

In [ ]:
#| export
def call_name(node):
    "Return dotted name text for an AST name, attribute, or call expression."
    parts = _name_parts(node)
    return ".".join(parts) if parts else None

In [ ]:
#| export
def short_call_name(node, default=None):
    """Return the final dotted component of a call/name node, or `default`."""
    name = call_name(node)
    return default if name is None else symbol_short_name(name)

In [ ]:
#| export
def is_definition_node(node):
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [ ]:
#| export
def node_start_line(node):
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [ ]:
#| export
def is_export_directive(line):
    return re.match(r"^\s*#\|\s*(export|exports|exporti)(\s|$)", line) is not None

In [ ]:
#| export
def _is_export_gap(lines):
    return bool(lines) and all((not line.strip()) or is_export_directive(line) for line in lines)

In [ ]:
#| export
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

In [ ]:
#| export
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export:
        cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [ ]:
#| export
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

In [ ]:
#| export
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [ ]:
#| export
def cell_source(cell):
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

In [ ]:
#| export
def parse_code_cell(cell):
    """Parse a code cell after removing nbdev directives, or return `None`."""
    if getattr(cell, 'cell_type', None) != 'code': return None
    try: return ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return None

### Cell metadata and semantic classes

Once cells can be parsed, the tools need names for what they *are doing*. A code cell with output is an example; a code cell with an `assert` is a test; a markdown cell with prose is documentation. Those labels power filters such as `type=tests` and help review tools talk about notebooks in human terms.

### Domain entities

These tiny models give the project names for the shapes it already manipulates: cells, chapters, notebooks, and symbols. The classes stay thin; most behavior is patched onto them so each concept can grow near the helper that understands it.

### Typed cell vocabularies

`CellType` names the three notebook cell kinds, `SemanticType` names the reader-facing role a cell plays, and `Directive` names the nbdev directives an editor can stamp onto a cell. Enums keep the rest of the project from passing bare strings around, and let context drop directive lines because the type already carries that meaning.

In [ ]:
#| export
class _StrEnum(str, Enum):
    "A string enum that compares and renders as its value."
    def __str__(self): return self.value
    def __format__(self, spec): return format(self.value, spec)
    @classmethod
    def coerce(cls, value):
        "Return the member matching `value`, or `UNKNOWN`."
        return next((m for m in cls if m.value == value), cls.UNKNOWN)


class CellType(_StrEnum):
    "The notebook cell kinds nbskill understands."
    CODE = "code"; MARKDOWN = "markdown"; RAW = "raw"; UNKNOWN = "unknown"


class SemanticType(_StrEnum):
    "The reader-facing role of a cell, used by context and review tools."
    MARKDOWN = "markdown"; DOCS = "docs"; SECTION = "section"; EXPORT = "exported code"
    IMPORT = "import"; PRIVATE = "private"; TEST = "test"; EXAMPLE = "example"
    HIDDEN = "hidden"; CODE = "code"; RAW = "raw"; UNKNOWN = "unknown"


class Directive(_StrEnum):
    "nbdev cell directives an editor can stamp onto a cell."
    EXPORT = "export"; NO_TEST = "eval: false"; HIDE = "hide"
    @property
    def line(self):
        "Return the `#| ...` directive line."
        return f"#| {self.value}"


def directive_lines(directives):
    "Return `#| ...` lines for `Directive`s given as members, names, or a comma string."
    if not directives: return []
    if isinstance(directives, Directive): directives = [directives]
    elif isinstance(directives, str): directives = directives.split(",")
    lines = []
    for d in directives:
        if isinstance(d, Directive): lines.append(d.line); continue
        key = str(d).strip()
        if not key: continue
        match = next((m for m in Directive if key.lower() in (m.name.lower(), m.value)), None)
        lines.append(match.line if match else f"#| {key}")
    return lines


def apply_directives(source, directives):
    "Prepend any missing `#| ...` directive lines to `source`."
    lines = directive_lines(directives)
    if not lines: return source
    existing = source.splitlines()
    missing = [line for line in lines if line not in existing]
    if not missing: return source
    return "\n".join([*missing, *existing])

In [ ]:
#| export
@dataclass
class Cell:
    "A focused view of one notebook cell plus its optional location."
    cell: object
    idx: object = None
    path: object = None

    def __post_init__(self):
        if self.path is not None: self.path = str(self.path)


@dataclass
class Chapter:
    "A contiguous heading-owned span inside a notebook."
    title: str
    start: int
    end: int
    cells: object = field(default_factory=list)

    def __post_init__(self):
        self.title = str(self.title)
        self.start = int(self.start)
        self.end = int(self.end)
        self.cells = list(self.cells or [])


@dataclass
class Notebook:
    "A notebook plus the path it was read from."
    nb: object
    path: object = None

    def __post_init__(self):
        if self.path is not None: self.path = str(self.path)


@dataclass
class NotebookSymbol:
    "A named notebook symbol plus optional placement and graph data."
    name: str
    path: object = None
    cell_id: str = ''
    cell_idx: object = None
    module: str = ''
    kind: str = ''
    data: object = field(default_factory=dict)

    def __post_init__(self):
        self.name = str(self.name)
        if self.path is not None: self.path = str(self.path)
        self.cell_id = self.cell_id or ''
        self.module = self.module or ''
        self.kind = self.kind or ''
        self.data = self.data or {}


# Historical names kept as aliases so older imports keep working.
NotebookCell = Cell
NotebookChapter = Chapter
NotebookDocument = Notebook

`xml_escape` and `xml_attrs` keep model-to-XML snippets boring and predictable. They are intentionally small because the XML is only for readable LLM context, not a general document writer.

In [ ]:
#| export
def xml_escape(text):
    "Escape text for simple XML-shaped context snippets."
    return (str(text).replace("&", "&amp;").replace("<", "&lt;")
            .replace(">", "&gt;").replace('"', "&quot;"))


def xml_attrs(**kwargs):
    "Render non-empty keyword values as XML attributes."
    attrs = []
    for key, value in kwargs.items():
        if value in (None, "", ()): continue
        attrs.append(f'{key}="{xml_escape(value)}"')
    return " ".join(attrs)

In [ ]:
{
    "escaped": xml_escape("<cell>"),
    "attrs": xml_attrs(kind="cell", title="<demo>"),
}

{'escaped': '&lt;cell&gt;', 'attrs': 'kind="cell" title="&lt;demo&gt;"'}

In [ ]:
#| hide
assert xml_escape("<cell>") == "&lt;cell&gt;"
assert xml_attrs(kind="cell", title="<demo>") == 'kind="cell" title="&lt;demo&gt;"'

In [ ]:
#| export
@patch(cls_method=True)
def from_source(cls: NotebookCell, source, default_type="code", idx=None, path=None):
    "Build a cell model from editable source text."
    return cls(parse_one_cell(source, default_type=default_type), idx=idx, path=path)

In [ ]:
#| export
@patch(as_prop=True)
def id(self: Cell):
    "Return the notebook cell id."
    return getattr(self.cell, "id", "")


@patch(as_prop=True)
def cell_type(self: Cell):
    "Return the cell type as a `CellType`."
    return CellType.coerce(getattr(self.cell, "cell_type", "") or "unknown")

In [ ]:
#| export
@patch(as_prop=True)
def source(self: NotebookCell):
    "Return the cell source as one string."
    return cell_source(self.cell)

In [ ]:
#| export
@patch(as_prop=True)
def code(self: NotebookCell):
    "Return source without nbdev directives."
    return source_without_directives(self.source)


@patch(as_prop=True)
def metadata(self: NotebookCell):
    "Return mutable cell metadata."
    return cell_metadata(self.cell)

In [ ]:
#| export
@patch
def directives(self: NotebookCell):
    "Return nbdev directive lines without the leading marker."
    return [line.strip()[2:].strip() for line in self.source.splitlines() if line.strip().startswith("#|")]


@patch
def directive(self: NotebookCell, name=None):
    "Return the first directive, or the value for one directive name."
    for value in self.directives():
        if name is None: return value
        if value == name: return ""
        if value.startswith(f"{name} "): return value[len(name):].strip()
        if value.startswith(f"{name}:"): return value[len(name) + 1:].strip()
    return None

In [ ]:
#| export
@patch
def class_names(self: NotebookCell):
    "Return semantic class names for the cell."
    return cell_class_names(self.cell)


@patch
def heading_title(self: NotebookCell, levels=(2,)):
    "Return the first markdown heading title whose level is allowed."
    if self.cell_type != "markdown": return None
    level_set = {int(level) for level in levels}
    for line in self.source.splitlines():
        match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line.strip())
        if match and len(match.group(1)) in level_set: return match.group(2).strip()
    return None

In [ ]:
#| export
@patch
def prefix(self: NotebookCell, show_ids=False):
    "Return the compact rendered cell prefix used in context output."
    return f"Cell id={self.id}: {self.cell_type}"

In [ ]:
#| export
@patch
def to_record(self: Cell):
    "Return a small structured record for this cell."
    return {
        "cell_id": self.id,
        "cell_idx": self.idx,
        "cell_type": self.cell_type,
        "path": self.path,
        "directives": self.directives(),
        "class_names": self.class_names(),
        "source": self.source,
    }


@patch
def to_xml(self: Cell, include_output=False):
    "Return an XML-shaped cell view for LLM context; directives become the semantic type."
    attrs = xml_attrs(id=self.id, idx=self.idx, type=self.cell_type, path=self.path,
                      semantic=self.semantic_type(), classes=" ".join(self.class_names()))
    body = [f"<source>{xml_escape(self.code)}</source>"]
    if include_output and _has_cell_output(self.cell):
        body.append(f"<outputs>{xml_escape(json.dumps(_cell_outputs(self.cell), default=str))}</outputs>")
    return f"<cell {attrs}>\n" + "\n".join(body) + "\n</cell>"

In [ ]:
#| export
@patch(cls_method=True)
def from_span(cls: NotebookChapter, span, cells=None):
    "Build a chapter model from a span dictionary."
    return cls(span["title"], span["start"], span["end"], cells=cells)


@patch(cls_method=True)
def all(cls: NotebookChapter, cells, levels=(2,), fallback=None):
    "Return all chapter models opened by markdown headings."
    starts = [(idx, NotebookCell(cell, idx=idx).heading_title(levels=levels)) for idx, cell in enumerate(cells)]
    starts = [(idx, title) for idx, title in starts if title]
    if not starts and fallback is not None: return [cls(fallback, 0, len(cells), cells=cells)]
    chapters = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        chapters.append(cls(title, start, end, cells=cells))
    return chapters

In [ ]:
#| export
@patch(as_prop=True)
def body(self: NotebookChapter):
    "Return cells inside this chapter span."
    return self.cells[self.start:self.end]

In [ ]:
#| export
@patch
def to_span(self: NotebookChapter):
    "Return the dictionary span shape used by existing APIs."
    return {"title": self.title, "start": self.start, "end": self.end}

In [ ]:
#| export
@patch
def to_record(self: NotebookChapter):
    "Return structured chapter metadata."
    return {**self.to_span(), "cell_count": max(0, self.end - self.start)}

In [ ]:
#| export
@patch
def to_xml(self: NotebookChapter, include_cells=True):
    "Return an XML-shaped chapter view for LLM context."
    attrs = xml_attrs(title=self.title, start=self.start, end=self.end)
    if not include_cells: return f"<chapter {attrs} />"
    cells = [NotebookCell(cell, idx=self.start + offset).to_xml() for offset, cell in enumerate(self.body)]
    return f"<chapter {attrs}>\n" + "\n".join(cells) + "\n</chapter>"

In [ ]:
#| export
@patch(cls_method=True)
def from_path(cls: NotebookDocument, path):
    "Read a notebook path into a notebook document model."
    from fastcore.nbio import read_nb
    return cls(read_nb(path), path=path)


@patch(as_prop=True)
def cells(self: NotebookDocument):
    "Return notebook cells as cell models."
    return [NotebookCell(cell, idx=idx, path=self.path) for idx, cell in enumerate(getattr(self.nb, "cells", []))]

In [ ]:
#| export
@patch
def cell(self: NotebookDocument, cell_id):
    "Return one cell model by stable id."
    idx, cell = find_cell_by_id(getattr(self.nb, "cells", []), cell_id)
    return NotebookCell(cell, idx=idx, path=self.path)


@patch
def chapter(self: NotebookDocument, title, create=False):
    "Return one chapter model by title or regex."
    return NotebookChapter.from_span(one_chapter(getattr(self.nb, "cells", []), title, create=create), self.nb.cells)


@patch
def notebook_source_hash(self: NotebookDocument):
    "Return the notebook's ordered source hash."
    return notebook_hash(self.nb)


@patch
def commit(self: NotebookDocument, **kwargs):
    "Commit this notebook model through the shared atomic writer."
    return commit_notebook(self.path, self.nb, **kwargs)

The notebook domain model gives callers small, navigable objects: `Notebook.cell` finds a stable cell id, `Notebook.chapter` returns a dynamic `Chapter`, `Notebook.notebook_source_hash` summarizes ordered source, and `Notebook.commit` writes through the same atomic path as the editing tools.

In [ ]:
#| export
@patch
def chapters(self: NotebookDocument, levels=(2,), fallback=None):
    "Return notebook chapters as chapter models."
    return NotebookChapter.all(getattr(self.nb, "cells", []), levels=levels, fallback=fallback)


@patch
def to_xml(self: NotebookDocument, levels=(2,), include_cells=True):
    "Return an XML-shaped notebook view for LLM context."
    attrs = xml_attrs(path=self.path)
    chapters = [chapter.to_xml(include_cells=include_cells) for chapter in self.chapters(levels=levels, fallback="Notebook")]
    return f"<notebook {attrs}>\n" + "\n".join(chapters) + "\n</notebook>"

In [ ]:
#| export
@patch(cls_method=True)
def from_record(cls: NotebookSymbol, record, data=None):
    "Build a symbol model from a graph or definition record."
    return cls(
        record.get("symbol") or record.get("name", ""),
        path=record.get("path"),
        cell_id=record.get("cell_id", ""),
        cell_idx=record.get("cell_idx"),
        module=record.get("module", ""),
        kind=record.get("kind", ""),
        data=data,
    )

In [ ]:
#| export
@patch
def to_record(self: NotebookSymbol):
    "Return a small structured record for this symbol."
    return {
        "symbol": self.name,
        "path": self.path,
        "cell_id": self.cell_id,
        "cell_idx": self.cell_idx,
        "module": self.module,
        "kind": self.kind,
    }

In [ ]:
#| export
@patch
def to_xml(self: NotebookSymbol):
    "Return an XML-shaped symbol view for LLM context."
    attrs = xml_attrs(symbol=self.name, path=self.path, cell_id=self.cell_id,
                       cell_idx=self.cell_idx, module=self.module, kind=self.kind)
    return f"<symbol {attrs} />"

In [ ]:
#| export
_NBSKILL_METADATA_KEY = "nbskill"
_NBSKILL_CELL_METADATA_ALIASES = {
    "nbskill_executed_hash": "executed_hash",
    "nbskill_timeout_hash": "timeout_hash",
    "nbskill_timeout_seconds": "timeout_seconds",
}

In [ ]:
#| export
def cell_metadata(cell):
    meta = cell.get("metadata", None) if isinstance(cell, dict) else getattr(cell, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(cell, dict): cell["metadata"] = meta
        else: cell.metadata = meta
    return meta

In [ ]:
#| export
def notebook_metadata(nb):
    meta = nb.get("metadata", None) if isinstance(nb, dict) else getattr(nb, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(nb, dict): nb["metadata"] = meta
        else: nb.metadata = meta
    return meta


In [ ]:
#| export
def _nbskill_cell_metadata(cell, create=True):
    meta = cell_metadata(cell) if create else (cell.get("metadata", {}) if isinstance(cell, dict) else getattr(cell, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if not isinstance(info, dict):
        if not create:
            legacy = {
                new_key: meta[old_key]
                for old_key, new_key in _NBSKILL_CELL_METADATA_ALIASES.items()
                if isinstance(meta, dict) and old_key in meta
            }
            return legacy or None
        meta[_NBSKILL_METADATA_KEY] = {}
        info = meta[_NBSKILL_METADATA_KEY]
    if isinstance(meta, dict):
        for old_key, new_key in _NBSKILL_CELL_METADATA_ALIASES.items():
            if old_key in meta: info.setdefault(new_key, meta.pop(old_key))
    return info

In [ ]:
#| export
def _nbskill_notebook_metadata(nb, create=True):
    meta = notebook_metadata(nb) if create else (nb.get("metadata", {}) if isinstance(nb, dict) else getattr(nb, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if isinstance(info, dict): return info
    if not create: return None
    meta[_NBSKILL_METADATA_KEY] = {}
    return meta[_NBSKILL_METADATA_KEY]

In [ ]:
#| export
def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


In [ ]:
#| export
def _metadata_path(path):
    path = Path(path)
    try: return path.resolve().relative_to(Path.cwd().resolve()).as_posix()
    except (OSError, ValueError): return path.as_posix()


In [ ]:
#| export
def _default_exp_from_notebook(nb):
    for cell in getattr(nb, "cells", []):
        for line in cell_source(cell).splitlines():
            match = re.match(r"^\s*#\|\s*default_exp\s+(.+?)\s*$", line)
            if match: return match.group(1).strip()
    return None


In [ ]:
#| export
def exported_py_path(nb_path, nb=None):
    "Return the generated Python file path for an nbdev notebook, if it has one."
    nb_path = Path(nb_path)
    if nb is None:
        from fastcore.nbio import read_nb
        nb = read_nb(nb_path)
    default_exp = _default_exp_from_notebook(nb)
    if not default_exp: return None
    try:
        from nbdev.config import get_config
        lib_path = Path(get_config(nb_path.parent).lib_path)
    except Exception:
        lib_path = nb_path.parent.parent / default_exp.split(".", 1)[0]
    return lib_path / (default_exp.replace(".", "/") + ".py")


In [ ]:
assert source_without_directives("#| export\nx = 1\n#| default_exp demo") == "x = 1"
with write_demo_notebook("foundation_notebook_paths.ipynb") as nb_path:
    checkpoint = nb_path.parent / ".ipynb_checkpoints" / "skip.ipynb"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    write_nb(new_nb([mk_cell("assert True")]), checkpoint)
    try:
        paths = notebook_paths(nb_path.parent)
        assert nb_path in paths
        assert checkpoint not in paths
    finally:
        remove_demo_path(checkpoint.parent)
generated = demo_path("generated_owner_sample.py")
try:
    generated.write_text("# AUTOGENERATED! DO NOT EDIT! File to edit: ../00_foundation.ipynb.\n", encoding="utf-8")
    assert generated_owner(generated).name == "00_foundation.ipynb"
    assert file_line_count(generated) == 1
finally:
    remove_demo_path(generated)
assert cap_text("abcdef", limit=3).startswith("abc")
assert cap_text("abcdef", max_output_chars=3)["truncated"] is True

In [ ]:
#| export
def stamp_export_metadata(nb, py_path):
    info = _nbskill_notebook_metadata(nb)
    info["exported_py_path"] = _metadata_path(py_path)
    info["exported_py_hash"] = file_hash(py_path)
    return nb


In [ ]:
#| export
def _nbdev_project_root(path):
    path = Path(path).expanduser()
    base = path.parent if path.suffix else path
    candidates = [base, *base.parents]
    for candidate in candidates:
        if _looks_like_nbdev_project(candidate): return candidate
    return None

In [ ]:
#| export
@contextmanager
def _temporary_cwd(path):
    previous = Path.cwd()
    os.chdir(path)
    try: yield
    finally: os.chdir(previous)

### Project-scoped nbdev export

`run_nbdev_export_from_project` and `export_notebook` deliberately run nbdev export in an isolated process under a project-scoped lock. This looks heavier than calling `nbdev_export` directly, but it protects the MCP server when several repositories on the same computer are edited at the same time.

The two failures this avoids are subtle. First, nbdev export depends on project context; changing the server process cwd with `os.chdir` is global state, so concurrent exports from different projects can race. Second, an MCP client timeout does not stop a Python worker thread, so a timed-out edit can keep exporting in the background and consume memory. A child process is a boundary the caller can terminate, while the lock only serializes exports for the same project. Different projects get different lock files and can still make progress independently.

In [ ]:
#| export
def _nbdev_export_lock_path(path):
    root = Path(path).expanduser().resolve(strict=False)
    digest = hashlib.sha256(str(root).encode("utf-8")).hexdigest()[:24]
    base = Path(os.environ.get("NBSKILL_LOCK_DIR") or tempfile.gettempdir()) / "nbskill-export-locks"
    base.mkdir(parents=True, exist_ok=True)
    return base / f"{digest}.lock"

In [ ]:
#| export
@contextmanager
def _nbdev_export_lock(path):
    lock_path = _nbdev_export_lock_path(path)
    try: import fcntl
    except ImportError:
        yield
        return
    with lock_path.open("a+", encoding="utf-8") as handle:
        handle.seek(0)
        handle.truncate()
        handle.write(str(Path(path).expanduser().resolve(strict=False)) + "\n")
        handle.flush()
        fcntl.flock(handle.fileno(), fcntl.LOCK_EX)
        try: yield
        finally: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

In [ ]:
#| hide
with (
    demo_path_context("foundation_export_project_a") as project_a,
    demo_path_context("foundation_export_project_b") as project_b,
):
    lock_a = _nbdev_export_lock_path(project_a)
    lock_b = _nbdev_export_lock_path(project_b)
    assert lock_a != lock_b
    assert lock_a.parent == lock_b.parent

In [ ]:
#| export
def _nbdev_export_process_worker(path, cwd, result_queue):
    out, err = StringIO(), StringIO()
    try:
        if cwd is not None: os.chdir(cwd)
        from nbdev.doclinks import nbdev_export
        with redirect_stdout(out), redirect_stderr(err): nbdev_export(path=path)
        result_queue.put(dict(ok=True, stdout=out.getvalue(), stderr=err.getvalue()))
    except BaseException as exc:
        result_queue.put(dict(
            ok=False, error_type=type(exc).__name__, error=str(exc),
            traceback=traceback.format_exc(), stdout=out.getvalue(), stderr=err.getvalue(),
        ))

In [ ]:
#| export
def _nbdev_export_process(path, cwd=None):
    from nbskill.foundation import _nbdev_export_process_worker as worker
    ctx = multiprocessing.get_context("spawn")
    result_queue = ctx.Queue()
    proc = ctx.Process(target=worker, args=(str(path), str(cwd) if cwd is not None else None, result_queue))
    proc.start()
    proc.join()
    try: payload = result_queue.get(timeout=1)
    except queue.Empty:
        payload = dict(ok=False, error_type="ProcessError", error=f"nbdev export process exited without a result payload; exit code {proc.exitcode}")
    if proc.exitcode != 0 or not payload.get("ok", True):
        lines = [f"{payload.get('error_type', 'Error')}: {payload.get('error', '')}".rstrip()]
        if payload.get("traceback"): lines.append(payload["traceback"].rstrip())
        captured = "\n".join(item.rstrip() for item in (payload.get("stdout"), payload.get("stderr")) if item)
        if captured: lines.append("Captured output:\n" + captured)
        raise RuntimeError("\n\n".join(line for line in lines if line))

In [ ]:
#| export
def run_nbdev_export_from_project(nb_path):
    "Run nbdev export in a project-scoped process without changing server cwd."
    nb_path = Path(nb_path).expanduser()
    project_root = _nbdev_project_root(nb_path)
    if project_root is None:
        export_path = nb_path.resolve(strict=False)
        with _nbdev_export_lock(export_path.parent): _nbdev_export_process(export_path, cwd=export_path.parent)
        return

    resolved = nb_path.resolve(strict=False)
    try: export_path = resolved.relative_to(project_root)
    except ValueError: export_path = resolved
    with _nbdev_export_lock(project_root): _nbdev_export_process(export_path, cwd=project_root)

In [ ]:
#| export
def export_notebook(nb, nb_path, writer=write_nb):
    "Export an nbdev notebook and stamp export metadata when possible."
    nb_path = Path(nb_path)
    py_path = exported_py_path(nb_path, nb)
    if py_path is None: return None
    run_nbdev_export_from_project(nb_path)
    if py_path.exists():
        stamp_export_metadata(nb, py_path)
        writer(nb, nb_path)
    return py_path

In [ ]:
#| eval: false
# Export a real nbdev notebook without changing the MCP server's cwd.
run_nbdev_export_from_project("nbs/00_foundation.ipynb")

In [ ]:
#| hide
with (
    demo_path_context("foundation_export_lock") as lock_root,
    demo_path_context("foundation_export_lock_other") as other_root,
):
    first = _nbdev_export_lock_path(lock_root)
    assert first == _nbdev_export_lock_path(lock_root)
    assert first != _nbdev_export_lock_path(other_root)
    with _nbdev_export_lock(lock_root):
        assert first.exists()

In [ ]:
#| export
def ensure_cell_ids(nb):
    "Add nbdev-style ids to notebook cells missing `id`, mutating and returning `nb`."
    for cell in getattr(nb, "cells", nb):
        if "id" not in cell: cell["id"] = rtoken_hex(4)
    return nb


def notebook_hash(nb):
    "Return a stable source hash for a notebook's ordered cell contents."
    payload = chr(30).join(f"{getattr(cell, 'id', '')}:{cell_source(cell)}" for cell in getattr(nb, "cells", []))
    return source_hash(payload)


def _path_inside_cwd(path):
    try: Path(path).expanduser().resolve(strict=False).relative_to(Path.cwd().resolve())
    except (OSError, ValueError): return False
    return True


def _restore_file_bytes(path, data):
    path = Path(path)
    if data is None:
        path.unlink(missing_ok=True)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)


def _readback_cell_hashes(nb, cell_ids):
    wanted = {cell_id for cell_id in (cell_ids or []) if cell_id}
    return {
        getattr(cell, "id", ""): source_hash(cell_source(cell))
        for cell in getattr(nb, "cells", [])
        if not wanted or getattr(cell, "id", "") in wanted
    }


def _clear_changed_outputs(nb, cell_ids=None):
    wanted = {cell_id for cell_id in (cell_ids or []) if cell_id}
    for cell in getattr(nb, "cells", []):
        if wanted and getattr(cell, "id", "") not in wanted: continue
        clear_outputs(cell)


def commit_notebook(
    path,
    trial,
    before=None,
    affected_cell_ids=None,
    validate_code=True,
    dry_run=False,
    export=True,
    writer=write_nb,
):
    "Validate, stamp, write, export, and rollback one notebook as one commit."
    from fastcore.nbio import read_nb

    path = Path(path)
    before = before if before is not None else (read_nb(path) if path.exists() else new_nb([]))
    missing_cell_ids = any("id" not in cell for cell in getattr(trial, "cells", []))
    ensure_cell_ids(before)
    ensure_cell_ids(trial)
    before_hash = notebook_hash(before)
    planned_hash = notebook_hash(trial)
    changed = before_hash != planned_hash or missing_cell_ids
    affected = [cell_id for cell_id in dict.fromkeys(affected_cell_ids or []) if cell_id]
    if validate_code and changed: validate_code_cells(trial.cells)

    exported = False
    export_no_drift = False
    py_path = exported_py_path(path, trial)
    if changed and not dry_run:
        _clear_changed_outputs(trial, affected or None)
        stamp_notebook_metadata(trial)
        before_path_bytes = path.read_bytes() if path.exists() else None
        before_py_bytes = py_path.read_bytes() if py_path is not None and py_path.exists() else None
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            writer(trial, path)
            if export and py_path is not None and _path_inside_cwd(path):
                run_nbdev_export_from_project(path)
                if py_path.exists():
                    stamp_export_metadata(trial, py_path)
                    writer(trial, path)
                    exported = True
        except Exception:
            _restore_file_bytes(path, before_path_bytes)
            if py_path is not None: _restore_file_bytes(py_path, before_py_bytes)
            raise
        readback = read_nb(path)
        if exported and py_path is not None and py_path.exists():
            info = _nbskill_notebook_metadata(readback, create=False) or {}
            export_no_drift = info.get("exported_py_hash") == file_hash(py_path)
        after_hash = notebook_hash(readback)
        readback_hashes = _readback_cell_hashes(readback, affected)
    else:
        after_hash = before_hash if dry_run else planned_hash
        readback_hashes = _readback_cell_hashes(before if dry_run else trial, affected)

    return {
        "changed": changed,
        "dry_run": dry_run,
        "affected_cell_ids": affected,
        "before_hash": before_hash,
        "planned_hash": planned_hash,
        "after_hash": after_hash,
        "readback_hashes": readback_hashes,
        "exported": exported,
        "exported_py_path": str(py_path) if py_path is not None else "",
        "export_no_drift": export_no_drift,
    }

`notebook_hash` and `commit_notebook` form the shared commit primitive. Callers build a trial notebook, pass the original as `before`, and get read-back hashes after the write succeeds. Changed code-cell outputs are cleared before the notebook is committed.

In [ ]:
#| export
def _fresh_semantic_metadata(cell):
    info = _nbskill_cell_metadata(cell, create=False)
    if not info: return None
    if info.get("cell_type") != getattr(cell, "cell_type", None): return None
    types = info.get("semantic_types")
    if not isinstance(types, list): return None
    normalized = tuple("example_cell" if str(item) == "exploration_cell" else str(item) for item in types)
    if getattr(cell, "cell_type", None) != "code" and "unclean_cell" in normalized: return None
    semantic = tuple(name for name in normalized if name not in {"exported_code", "unclean_cell"})
    if "unclean_cell" in normalized and len(semantic) <= 1: return None
    return normalized


In [ ]:
#| export
def parse_one_cell(text, default_type="code"):
    if isinstance(text, (list, tuple)):
        if len(text) != 1:
            cli_error("update_cell expects exactly one replacement cell; use write_nb or batch_edit_nb for multi-cell edits")
        return _coerce_cell(text[0], default_type)
    blocks = _split_blocks(text)
    if len(blocks) != 1:
        cli_error("update_cell expects exactly one replacement cell; remove standalone '---' separators or use write_nb/batch_edit_nb for multi-cell edits")
    return _cell_from_block(blocks[0], default_type)

In [ ]:
#| export
def find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == cell_id]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error(f"No cell has id {cell_id!r}")
    cli_error(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def find_cell_by_text(cells, old_str):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if old_str in cell_source(cell)]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error("old_str did not match any cell")
    idxs = ", ".join(str(idx) for idx, _ in matches)
    cli_error(f"old_str matched multiple cells: {idxs}. Use --cell_id or a more specific old_str.")

In [ ]:
#| export
def replace_cell(nb, idx, new_cell):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None: new_cell.id = old_id
    nb.cells[idx] = new_cell

In [ ]:
#| export
def clear_outputs(cell):
    if getattr(cell, "cell_type", None) == "code":
        cell.outputs = []
        cell.execution_count = None
    return cell

In [ ]:
#| export
def _looks_like_multiline_cli_text(text):
    if not isinstance(text, str) or "\\n" not in text: return False
    stripped = text.lstrip().lower()
    if stripped.startswith(("%%code\\n", "%%markdown\\n", "%%md\\n", "%%raw\\n")): return True
    if "\\n---\\n" in text: return True
    return "\\n    " in text or "\\n\t" in text


In [ ]:
#| export
def _decode_cli_newlines(text):
    return text.replace("\\n", "\n") if _looks_like_multiline_cli_text(text) else text


In [ ]:
#| export
def load_cells_text(cells="", cells_file=None, decode_newlines=True):
    if cells_file:
        if cells: raise ValueError("Use either cells or cells_file, not both")
        return Path(cells_file).expanduser().read_text(encoding="utf-8")
    if cells == "-": return sys.stdin.read()
    return _decode_cli_newlines(cells) if decode_newlines else cells


In [ ]:
#| export
def _should_validate_python(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("%", "!")): return False
    return bool(source.strip())

In [ ]:
#| export
def _format_syntax_error(source, err, cell_idx):
    lines = source.splitlines()
    line = lines[err.lineno - 1] if err.lineno and 0 < err.lineno <= len(lines) else ""
    pointer = " " * max((err.offset or 1) - 1, 0) + "^" if line else ""
    msg = [f"Invalid Python in new code cell {cell_idx}: {err.msg} at line {err.lineno}, column {err.offset}"]
    if line: msg += [line, pointer]
    msg.append("Tip: shell quoting can turn backslash-n escapes into real newlines inside Python strings. Use --cells_file PATH or cells=- for complex code.")
    return chr(10).join(msg)

In [ ]:
#| export
def validate_code_cells(cells):
    for idx, cell in enumerate(cells):
        cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
        if cell_type != "code": continue
        source = cell_source(cell)
        if not _should_validate_python(source): continue
        try: ast.parse(source)
        except SyntaxError as err:
            msg = _format_syntax_error(source, err, idx)
            if is_cli(): raise SystemExit(msg)
            raise ValueError(msg) from err

In [ ]:
#| export
def _cell_from_block(block, default_type="code"):
    lines = block.splitlines()
    marker = lines[0].strip().lower() if lines else ""
    cell_type = default_type
    if marker in {"%%markdown", "%%md"}: cell_type, lines = "markdown", lines[1:]
    elif marker == "%%code": cell_type, lines = "code", lines[1:]
    elif marker == "%%raw": cell_type, lines = "raw", lines[1:]
    return mk_cell("\n".join(lines), cell_type=cell_type)


In [ ]:
#| export
def parse_cells(cells, default_type="code"):
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = [_cell_from_block(block, default_type) for block in _split_blocks(cells)]
    return _split_symbol_cells(parsed)


### Naming cells by behavior

The reading and MCP layers present cells by meaning, not just by `code` or `markdown`. This section detects imports, private helpers, exported code, tests, examples, docs, section headers, and mixed cells so callers can filter notebooks at a useful level.

In [ ]:
#| export
def first_line(source):
    for line in source.splitlines():
        line = line.strip()
        if line: return line
    return ""

In [ ]:
#| export
def cell_prefix(idx, cell, show_ids=False): return NotebookCell(cell, idx=idx).prefix(show_ids=show_ids)

In [ ]:
#| export
def matches_filter(source, pattern):
    pattern = str(pattern)
    if pattern in source: return True
    try: return re.search(pattern, source, flags=re.MULTILINE) is not None
    except re.error: return False

In [ ]:
#| export
def is_exported_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    return any(is_export_directive(line) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _cell_outputs(cell): return list(getattr(cell, "outputs", []) or [])


In [ ]:
#| export
def _has_cell_output(cell): return bool(_cell_outputs(cell))


In [ ]:
#| export
def output_value_text(value):
    """Return a notebook output value as plain text."""
    if isinstance(value, list): return ''.join(map(str, value))
    if value is None: return ''
    return str(value)


def output_text(output):
    """Return a notebook output object as plain text."""
    output_type = output.get('output_type', '')
    if output_type == 'stream': return output_value_text(output.get('text', ''))
    if output_type == 'error':
        traceback = output.get('traceback') or []
        if traceback: return '\n'.join(map(str, traceback))
        return f"{output.get('ename', '')}: {output.get('evalue', '')}"
    data = output.get('data', {})
    for key in ('text/plain', 'text/markdown', 'text/html'):
        if key in data: return output_value_text(data.get(key))
    return ''


_output_text_function = output_text


def cell_output_text(cell):
    """Return all visible outputs for `cell` as compact plain text."""
    return ''.join(_output_text_function(output) for output in _cell_outputs(cell)).strip()

In [ ]:
#| export
def _code_tree(cell):
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None


In [ ]:
#| export
def _code_body(cell):
    tree = _code_tree(cell)
    return [] if tree is None else list(tree.body)


In [ ]:
#| export
def _is_import_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    body = _code_body(cell)
    return bool(body) and all(isinstance(node, (ast.Import, ast.ImportFrom)) for node in body)


In [ ]:
#| export
def _has_private_function(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    tree = _code_tree(cell)
    if tree is None: return False
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("_")
        for node in tree.body)


In [ ]:
#| export
def _has_test_marker(cell):
    source = cell_source(cell)
    tree = _code_tree(cell)
    if tree is None: return re.search(r"\b(assert|test_[A-Za-z0-9_]*)\b", source) is not None
    if any(isinstance(node, ast.Assert) for node in ast.walk(tree)): return True
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body)


In [ ]:
#| export
def _is_test_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and not _has_cell_output(cell) and _has_test_marker(cell)


In [ ]:
#| export
def _is_example_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and _has_cell_output(cell)


In [ ]:
#| export
def _is_section_header(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _is_docs_cell(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(line.strip() and not re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


In [ ]:
#| export
def _cell_base_class_names(cell):
    names = []
    if _is_import_cell(cell): names.append("import_cell")
    if _is_example_cell(cell): names.append("example_cell")
    if _is_test_cell(cell): names.append("test_cell")
    if _has_private_function(cell): names.append("private_code")
    if is_exported_code_cell(cell): names.append("exported_code")
    if _is_docs_cell(cell): names.append("docs_cell")
    if _is_section_header(cell): names.append("section_header")
    return tuple(names)


In [ ]:
#| export
def _semantic_code_class_names(cell):
    semantic = {"import_cell", "example_cell", "test_cell", "private_code"}
    return tuple(name for name in _cell_base_class_names(cell) if name in semantic)


In [ ]:
#| export
def _fallback_cell_class_name(cell):
    cell_type = getattr(cell, "cell_type", None)
    return f"{cell_type}_cell" if cell_type else "unknown_cell"


In [ ]:
#| export
def _computed_cell_class_names(cell):
    names = _cell_base_class_names(cell)
    if getattr(cell, "cell_type", None) == "code" and len(_semantic_code_class_names(cell)) > 1:
        return (*names, "unclean_cell")
    return names or (_fallback_cell_class_name(cell),)


In [ ]:
#| export
def _refresh_cell_metadata(cell):
    info = _nbskill_cell_metadata(cell)
    semantic_types = _computed_cell_class_names(cell)
    info["cell_type"] = getattr(cell, "cell_type", None)
    info["semantic_types"] = list(semantic_types)
    info.pop("source_hash", None)
    return cell


In [ ]:
#| export
def stamp_notebook_metadata(nb, exported_py_path=None):
    for cell in getattr(nb, "cells", []): _refresh_cell_metadata(cell)
    if exported_py_path is not None: stamp_export_metadata(nb, exported_py_path)
    return nb


In [ ]:
from fastcore.nbio import read_nb

with write_demo_notebook("commit_notebook_example.ipynb", cells=[mk_cell(chr(10).join(["value = 1", "print(value)"]))]) as commit_path:
    before_nb = read_nb(commit_path)
    trial_nb = read_nb(commit_path)
    print(notebook_hash(before_nb))
    trial_nb.cells[0].source = chr(10).join(["value = 2", "print(value)"])
    trial_nb.cells[0].outputs = [dict(output_type="stream", name="stdout", text="stale" + chr(10))]
    result = commit_notebook(commit_path, trial_nb, before=before_nb, affected_cell_ids=[trial_nb.cells[0].id], export=False)
    print(result["changed"], result["affected_cell_ids"])
    print(read_nb(commit_path).cells[0].outputs)

f4991148b6ca
True ['ef2c210c']
[]


In [ ]:
#| hide
from fastcore.nbio import read_nb

idless_cell = mk_cell("value = 1")
idless_cell.pop("id", None)
idless_nb = new_nb([])
idless_nb.cells.append(idless_cell)
assert "id" not in idless_nb.cells[0]
assert ensure_cell_ids(idless_nb) is idless_nb
assert idless_nb.cells[0].id and len(idless_nb.cells[0].id) == 8

with write_demo_notebook("commit_adds_cell_id_test.ipynb", cells=[]) as id_path:
    before_nb = read_nb(id_path)
    trial_cell = mk_cell("created = 1")
    trial_cell.pop("id", None)
    trial_nb = new_nb([])
    trial_nb.cells.append(trial_cell)
    result = commit_notebook(id_path, trial_nb, before=before_nb, export=False)
    readback_nb = read_nb(id_path)
    assert result["changed"] is True
    assert readback_nb.cells[0].id and len(readback_nb.cells[0].id) == 8

with write_demo_notebook("commit_notebook_test.ipynb", cells=[mk_cell("value = 1")]) as commit_path:
    before_nb = read_nb(commit_path)
    trial_nb = read_nb(commit_path)
    trial_nb.cells[0].source = "value = 2"
    trial_nb.cells[0].outputs = [dict(output_type="stream", name="stdout", text="stale" + chr(10))]
    result = commit_notebook(commit_path, trial_nb, before=before_nb, affected_cell_ids=[trial_nb.cells[0].id], export=False)
    readback_nb = read_nb(commit_path)
    assert result["changed"] is True
    assert result["after_hash"] == notebook_hash(readback_nb)
    assert result["readback_hashes"][trial_nb.cells[0].id] == source_hash(readback_nb.cells[0].source)
    assert readback_nb.cells[0].outputs == []

In [ ]:
#| export
def cell_class_names(cell): return _fresh_semantic_metadata(cell) or _computed_cell_class_names(cell)


In [ ]:
with write_demo_notebook("00_foundation_context.ipynb") as path:
    assert path.exists()
    assert is_valid_ipynb(path)
assert not path.exists()

`parse_cells` is the little translator between a friendly edit format and real notebook cells. The dashed line means "start a new cell", while `%%markdown` and `%%code` make the intended cell type explicit.

In [ ]:
_demo_text = "\n".join(["%%markdown", "## Demo", "---", "%%code", "value = 42"])
_demo_cells = parse_cells(_demo_text)
for idx, cell in enumerate(_demo_cells):
    print(cell_prefix(idx, cell))
    print("  type:", cell.cell_type)
    print("  first line:", first_line(cell_source(cell)))

Cell id=25790bac: markdown
  type: markdown
  first line: ## Demo
Cell id=38f08303: code
  type: code
  first line: value = 42


In [ ]:
#| export
def _normalize_cell_type_filter(value):
    if value is None: return None
    aliases = {
        "code": "code",
        "py": "code",
        "python": "code",
        "md": "markdown",
        "markdown": "markdown",
        "doc": "markdown",
        "docs": "markdown",
        "raw": "raw",
        "export": "exported_code",
        "exported": "exported_code",
        "exported_code": "exported_code",
        "import": "import_cell",
        "imports": "import_cell",
        "import_cell": "import_cell",
        "private": "private_code",
        "private_code": "private_code",
        "test": "test_cell",
        "tests": "test_cell",
        "test_cell": "test_cell",
        "example": "example_cell",
        "examples": "example_cell",
        "example_cell": "example_cell",
        "exploration": "example_cell",
        "explorations": "example_cell",
        "exploration_cell": "example_cell",
        "docs_cell": "docs_cell",
        "documentation": "docs_cell",
        "section": "section_header",
        "header": "section_header",
        "section_header": "section_header",
        "unclean": "unclean_cell",
        "unclean_cell": "unclean_cell",
    }
    normalized = set()
    for item in str(value).split(","):
        key = item.strip().lower()
        if not key: continue
        if key not in aliases:
            choices = ", ".join(sorted(set(aliases)))
            raise ValueError(f"Unknown cell_type {item!r}; use one of: {choices}")
        normalized.add(aliases[key])
    return normalized or None


In [ ]:
#| export
def cell_matches_type(cell, cell_type):
    wanted = _normalize_cell_type_filter(cell_type)
    if wanted is None: return True
    if getattr(cell, "cell_type", None) in wanted: return True
    return bool(set(cell_class_names(cell)) & wanted)


In [ ]:
#| export
@patch
def output_text(self: Cell):
    "Return all visible outputs for this cell as compact plain text."
    return cell_output_text(self.cell)


@patch
def semantic_type(self: Cell):
    "Return the reader-facing `SemanticType` for this cell."
    if self.cell_type == CellType.MARKDOWN: return SemanticType.MARKDOWN
    if self.cell_type != CellType.CODE: return SemanticType.coerce(self.cell_type)
    if cell_matches_type(self.cell, 'test'): return SemanticType.TEST
    if cell_matches_type(self.cell, 'example'): return SemanticType.EXAMPLE
    if is_exported_code_cell(self.cell): return SemanticType.EXPORT
    if re.search(r'^\s*assert\b', self.source, re.MULTILINE): return SemanticType.TEST
    if re.search(r'^\s*#\|\s*hide\b', self.source, re.MULTILINE): return SemanticType.HIDDEN
    return SemanticType.EXAMPLE


@patch
def context_record(self: Cell):
    "Return the context record shape used by read helpers, with directives stripped."
    return {
        'cell_id': self.id,
        'cell_idx': self.idx,
        'cell_type': self.cell_type,
        'semantic_type': self.semantic_type(),
        'source': self.code.strip(),
        'output': self.output_text(),
    }


output_text = _output_text_function

In [ ]:
#| export
def with_context(cells, items, include=False):
    if not include: return items

    idxs = {idx for idx, _ in items}
    for idx in list(idxs):
        prev = idx - 1
        while prev >= 0 and cells[prev].cell_type == "markdown":
            idxs.add(prev)
            prev -= 1

        nxt = idx + 1
        while nxt < len(cells):
            cell = cells[nxt]
            if cell.cell_type != "code" or is_exported_code_cell(cell): break
            idxs.add(nxt)
            nxt += 1
    return [(idx, cells[idx]) for idx in sorted(idxs)]


### Chapters as editing scopes

Chapters are the notebook-native version of a module section. A heading cell opens a span, and everything until the next heading belongs to that span. Write, execute, and read tools use these spans so users can say "the query language chapter" instead of counting cell indexes by hand.

In [ ]:
#| export
def heading_title(cell, levels=(2,)):
    "Return the first markdown heading title whose level is allowed."
    return NotebookCell(cell).heading_title(levels=levels)


def _chapter_title(cell):
    return heading_title(cell, levels=(2,))

In [ ]:
#| export
def chapter_spans(cells, levels=(2,), fallback=None):
    "Return contiguous notebook cell spans opened by markdown headings."
    return [chapter.to_span() for chapter in NotebookChapter.all(cells, levels=levels, fallback=fallback)]

In [ ]:
#| export
def _matching_chapters(cells, chapter=None):
    spans = chapter_spans(cells)
    if chapter is None: return spans
    return [span for span in spans if matches_filter(span["title"], chapter)]

In [ ]:
#| export
def chapter_index_set(cells, chapter):
    idxs = set()
    for span in _matching_chapters(cells, chapter):
        idxs.update(range(span["start"], span["end"]))
    return idxs

In [ ]:
#| export
def one_chapter(cells, chapter, create=False):
    matches = _matching_chapters(cells, chapter)
    if len(matches) == 1: return matches[0]
    if not matches and create:
        cells.append(mk_cell(f"## {chapter}", cell_type="markdown"))
        return dict(title=str(chapter), start=len(cells) - 1, end=len(cells))
    if not matches: raise ValueError(f"No chapter matches {chapter!r}")
    titles = ", ".join(f"{span['title']} ({span['start']}:{span['end']})" for span in matches)
    raise ValueError(f"Chapter {chapter!r} matches multiple chapters: {titles}")

In [ ]:
with write_demo_notebook("domain_model_example.ipynb", cells=[mk_cell("## Alpha", cell_type="markdown"), mk_cell("answer = 42")]) as model_path:
    model_doc = Notebook.from_path(model_path)
    model_cell = model_doc.cell(model_doc.nb.cells[1].id)
    model_chapter = model_doc.chapter("Alpha")
    print(model_cell.source)
    print(model_chapter.to_record())
    print(model_doc.notebook_source_hash())
    print(model_doc.commit(before=model_doc.nb, export=False)["changed"])

answer = 42
{'title': 'Alpha', 'start': 0, 'end': 2, 'cell_count': 2}
a584b5a3979c
False


In [ ]:
#| hide
with write_demo_notebook("domain_model_test.ipynb", cells=[mk_cell("## Alpha", cell_type="markdown"), mk_cell("answer = 42")]) as model_path:
    model_doc = Notebook.from_path(model_path)
    assert model_doc.cell(model_doc.nb.cells[1].id).source == "answer = 42"
    assert model_doc.chapter("Alpha").to_record()["cell_count"] == 2
    assert model_doc.notebook_source_hash() == notebook_hash(model_doc.nb)
    assert model_doc.commit(before=model_doc.nb, export=False)["changed"] is False

On a real notebook, chapter spans become a tiny table of contents with cell ranges. Here `index.ipynb` is being parsed by the foundation helpers that the reading tools themselves rely on.

In [ ]:
_index_cells = read_nb("index.ipynb").cells
for span in chapter_spans(_index_cells)[:5]: print(f"{span['title']}: cells {span['start']}..{span['end'] - 1}")

print("reading indexes:", sorted(chapter_index_set(_index_cells, "Reading"))[:5])

The problem this project solves: cells 2..2
How the notebooks fit together: cells 3..3
Production readiness: cells 4..5
A tiny notebook to work on: cells 6..7
Reading: choose the smallest useful context: cells 8..9
reading indexes: [8, 9]


In [ ]:
#| export
def _chapter_body_len(span):
    return max(span["end"] - span["start"] - 1, 0)

In [ ]:
#| export
def _chapter_body_slice(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    start, stop, step = target.indices(body_len)
    if step != 1: raise ValueError("chapter ranges do not support steps")
    return slice(body_start + start, body_start + stop)

In [ ]:
custom_map = demo_path("00_foundation_errors.json")
_old_failure_map = os.environ.get("NBSKILL_FAILURE_MAP")
try:
    os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
    assert failure_map_path() == custom_map
finally:
    if _old_failure_map is None: os.environ.pop("NBSKILL_FAILURE_MAP", None)
    else: os.environ["NBSKILL_FAILURE_MAP"] = _old_failure_map

In [ ]:

cells = parse_cells("%%markdown\n## Demo\n---\n%%code\nvalue = 42")
assert len(cells) == 2
assert cells[0].cell_type == "markdown"
assert cells[1].source == "value = 42"
one = parse_one_cell("%%code\nvalue = 99")
assert one.cell_type == "code"
assert one.source == "value = 99"
assert load_cells_text("%%code\\nvalue = 1") == "%%code\nvalue = 1"
assert load_cells_text("def f():\\n    return 1") == "def f():\n    return 1"
assert load_cells_text("value = 'a\\nb'") == "value = 'a\\nb'"
encoded = "def f():" + chr(92) + "n    return 1"
assert load_cells_text(encoded, decode_newlines=False) == encoded


In [ ]:
#| hide
cell_model = NotebookCell.from_source("#| export\ndef sample():\n    return 1", idx=2, path="demo.ipynb")
assert cell_model.id
assert cell_model.cell_type == "code"
assert cell_model.directive() == "export"
assert cell_model.code == "def sample():\n    return 1"
assert cell_model.to_record()["cell_idx"] == 2
assert "<cell" in cell_model.to_xml() and "sample" in cell_model.to_xml()

chapter_cells = [mk_cell("## One", cell_type="markdown"), mk_cell("x = 1"), mk_cell("## Two", cell_type="markdown")]
chapters = NotebookChapter.all(chapter_cells)
assert [chapter.title for chapter in chapters] == ["One", "Two"]
assert chapter_spans(chapter_cells) == [{"title": "One", "start": 0, "end": 2}, {"title": "Two", "start": 2, "end": 3}]
assert "<chapter" in chapters[0].to_xml()

document = NotebookDocument(new_nb(chapter_cells), path="demo.ipynb")
assert [cell.idx for cell in document.cells] == [0, 1, 2]
assert "<notebook" in document.to_xml()

symbol = NotebookSymbol.from_record({"symbol": "sample", "path": "demo.ipynb", "cell_id": cell_model.id, "cell_idx": 2, "kind": "function"})
assert symbol.to_record()["symbol"] == "sample"
assert "sample" in symbol.to_xml()

Here are the labels on toy cells. This is deliberately small, but it is the same classification that lets the reading and review tools distinguish examples from tests without asking a model to guess.

In [ ]:
_semantic_samples = {}
_semantic_samples["import"] = mk_cell("import os", cell_type="code")
_semantic_samples["private"] = mk_cell("def _helper():\n    pass", cell_type="code")
_semantic_samples["exported"] = mk_cell("#| export\ndef public():\n    pass", cell_type="code")
_semantic_samples["test"] = mk_cell("assert 1 == 1", cell_type="code")
_semantic_samples["plain"] = mk_cell("value = 1", cell_type="code")
_semantic_samples["docs"] = mk_cell("Some notes", cell_type="markdown")
_semantic_samples["example"] = mk_cell("print('hello notebook')", cell_type="code")
_semantic_samples["example"].outputs = [dict(output_type="stream", name="stdout", text="hello notebook\n")]

for label, cell in _semantic_samples.items(): print(f"{label:8} -> {', '.join(cell_class_names(cell))}")

import   -> import_cell
private  -> private_code
exported -> exported_code
test     -> test_cell
plain    -> code_cell
docs     -> docs_cell
example  -> example_cell


The assertions below are the boring-but-important contract. They pin down the edge cases: mixed header/prose markdown, exported import cells that are normal nbdev cells, true mixed semantic code, and export metadata that points back to the generated `.py` file.

In [ ]:
import_cell = _semantic_samples["import"]
private_cell = _semantic_samples["private"]
exported_cell = _semantic_samples["exported"]
test_cell = _semantic_samples["test"]
plain_code_cell = _semantic_samples["plain"]
example_cell = _semantic_samples["example"]
docs_cell = _semantic_samples["docs"]

expected_classes = [
    (import_cell, ("import_cell",)), (private_cell, ("private_code",)), (exported_cell, ("exported_code",)),
    (test_cell, ("test_cell",)), (plain_code_cell, ("code_cell",)), (example_cell, ("example_cell",)),
    (docs_cell, ("docs_cell",))]
for cell, expected in expected_classes: assert cell_class_names(cell) == expected

section_cell = mk_cell("## API", cell_type="markdown")
docs_header_cell = mk_cell("## API\nSome notes", cell_type="markdown")
exported_import_cell = mk_cell("#| export\nimport os", cell_type="code")
unclean_cell = mk_cell("assert True\ndef _helper():\n    return 1", cell_type="code")
assert cell_class_names(section_cell) == ("section_header",)
assert cell_class_names(docs_header_cell) == ("docs_cell", "section_header")
assert cell_class_names(exported_import_cell) == ("import_cell", "exported_code")
assert cell_class_names(unclean_cell) == ("test_cell", "private_code", "unclean_cell")
assert cell_matches_type(example_cell, "example")
assert cell_matches_type(example_cell, "exploration")

Metadata stamping is separate from classification: it records the generated Python artifact and caches the semantic labels that downstream tools can reuse.

In [ ]:
export_path = demo_path("00_foundation_export.py")
export_path.write_text("print('exported')\n", encoding="utf-8")
stamped_nb = stamp_notebook_metadata(new_nb([exported_cell]), exported_py_path=export_path)
stamped_cell = stamped_nb.cells[0]
cell_info = stamped_cell.metadata["nbskill"]
notebook_info = notebook_metadata(stamped_nb)["nbskill"]
assert cell_info["cell_type"] == "code"
assert isinstance(cell_info["semantic_types"], list)
assert "source_hash" not in cell_info
assert notebook_info["exported_py_hash"] == file_hash(export_path)
assert notebook_info["exported_py_path"].endswith("00_foundation_export.py")
assert cell_class_names(stamped_cell) == ("exported_code",)
stamped_cell.metadata["nbskill"]["semantic_types"] = ["stored_type"]
assert cell_class_names(stamped_cell) == ("stored_type",)
stamped_cell.source += chr(10)
assert cell_class_names(stamped_cell) == ("stored_type",)
remove_demo_path(export_path)

Path('nbs/data/00_foundation_export.py')

Fastcore moved CLI detection from the private `_in_call_parse` context var to `is_cli()`. nbskill pins fastcore to the current release that provides `is_cli()`, so `cli_return`, `cli_error`, and validation errors use that public helper directly.

In [ ]:
#| hide
assert cli_return("direct") == "direct"
try:
    cli_error("direct failure")
except ValueError as err:
    assert str(err) == "direct failure"
else:
    raise AssertionError("cli_error should raise ValueError outside CLI calls")